# Assignment 1: The Headline Test — work
Data are **illustrative teaching data**, not actual PBS observations (see `data_label`).

## 1. Predictions (written BEFORE running code)
Fill these in from your `1_attempt.md` and re-run. `None` = not yet filled in.

In [1]:
PRED_SHAPE   = None      # e.g. (rows, cols) from your attempt file
PRED_N_ITEMS = None
PRED_N_DATES = None

## 2. Run and compare

In [2]:
import pandas as pd
d = pd.read_csv('teaching_prices_clean.csv', parse_dates=['date'])
actual = {'shape': d.shape, 'distinct items': d['item'].nunique(), 'distinct dates': d['date'].nunique()}
pred   = {'shape': PRED_SHAPE, 'distinct items': PRED_N_ITEMS, 'distinct dates': PRED_N_DATES}
for k in actual:
    print(f"{k:15} prediction={pred[k]!s:8} actual={actual[k]!s:8}", 'MATCH' if pred[k]==actual[k] else ('(fill in prediction)' if pred[k] is None else 'DIFFERENT'))

shape           prediction=None     actual=(96, 6)  (fill in prediction)
distinct items  prediction=None     actual=4        (fill in prediction)
distinct dates  prediction=None     actual=12       (fill in prediction)


In [3]:
print(d.dtypes); print()
print(d.isna().sum().sum(), 'missing values;', d.duplicated(['date','item','city']).sum(), 'duplicate date-item-city rows')
print(list(d['item'].unique()), list(d['city'].unique()), list(d['unit'].unique()))
print(d['date'].min().date(), 'to', d['date'].max().date(), '(weekly)')
print(d.groupby(['item','city']).size().unstack())   # balanced panel?

date          datetime64[us]
item                     str
city                     str
price                float64
unit                     str
data_label               str
dtype: object

0 missing values; 0 duplicate date-item-city rows
['Staple A', 'Oil B', 'Pulse C', 'Sweetener D'] ['City 1', 'City 2'] ['PKR per kg (illustrative)']
2026-01-01 to 2026-03-19 (weekly)
city         City 1  City 2
item                       
Oil B            12      12
Pulse C          12      12
Staple A         12      12
Sweetener D      12      12


## 3. Question 1 — claim: “Everything costs more.”
**Question:** Between 1 Jan and 19 Mar 2026, by what percentage did each item's price change in each city, and did it rise every week?  
**Variable needed:** `price` by `item`, `city`, `date`.

In [4]:
p = d.pivot_table(index='date', columns=['item','city'], values='price')
change = ((p.iloc[-1]/p.iloc[0]-1)*100).round(1).unstack()
change.columns.name = None
print('% change, first to last date'); print(change)
wk = p.diff().iloc[1:]
print('\nWeeks with a price FALL (out of', len(wk), 'weekly changes):')
print((wk<0).sum().unstack())

% change, first to last date
             City 1  City 2
item                       
Oil B          15.4    15.1
Pulse C         3.8     3.6
Staple A       13.4    12.6
Sweetener D    19.8    19.0

Weeks with a price FALL (out of 11 weekly changes):
city         City 1  City 2
item                       
Oil B             0       0
Pulse C           4       4
Staple A          5       5
Sweetener D       4       4


In [5]:
import matplotlib.pyplot as plt
ax = p.xs('City 1', axis=1, level='city').plot(figsize=(7,4), marker='o', title='Weekly price, City 1 (illustrative data)')
ax.set_ylabel('PKR per kg (illustrative)'); plt.tight_layout(); plt.savefig('price_trends_city1.png', dpi=110)

## 4. Question 2 — claim: “Staples are out of reach.”
**Question:** Did Staple A's price rise faster or slower than the other items, and does that differ between cities?  
**Variable needed:** `price` for Staple A vs the other items, by `city`.  
**Cannot test:** “out of reach” needs incomes/wages or a budget share — none in this file.

In [6]:
rank = change.stack().rename('pct_change').reset_index().sort_values('pct_change', ascending=False)
print(rank.to_string(index=False))
g = d.pivot_table(index=['date','item'], columns='city', values='price')
gap = g['City 2'] - g['City 1']
print('\nCity 2 minus City 1 (PKR/kg), by item:'); print(gap.groupby('item').agg(['min','max']))
a = p['Staple A']['City 1']
print('\nStaple A, City 1: first', a.iloc[0], 'last', a.iloc[-1], 'absolute change', a.iloc[-1]-a.iloc[0])

       item level_1  pct_change
Sweetener D  City 1        19.8
Sweetener D  City 2        19.0
      Oil B  City 1        15.4
      Oil B  City 2        15.1
   Staple A  City 1        13.4
   Staple A  City 2        12.6
    Pulse C  City 1         3.8
    Pulse C  City 2         3.6

City 2 minus City 1 (PKR/kg), by item:
             min  max
item                 
Oil B        6.0  6.0
Pulse C      6.0  6.0
Staple A     6.0  6.0
Sweetener D  6.0  6.0

Staple A, City 1: first 101.0 last 114.5 absolute change 13.5


## 5. Reading the results
- All four items are higher on 19 Mar than on 1 Jan in both cities, so “everything costs more” holds *in this file* — but by very different amounts: Sweetener D ≈ +19–20 %, Oil B ≈ +15 %, Staple A ≈ +13 %, Pulse C only ≈ +4 %.
- Rises are not smooth: some items fell in some weeks, so the choice of start and end dates matters.
- City 2 is exactly 6 PKR/kg above City 1 for every item and date: the cities move in lockstep, and the % changes differ only because of the different base.
- Staple A rose about 13 % — mid-table, not the fastest. The data say nothing about whether that is *affordable*.